In [1]:
#captions
#import libraries

import pandas as pd
import numpy as np

In [2]:
#captions
#import final Alteryx output dataset
#replace the file name below if your exported CSV has a different name

df = pd.read_csv("Australian Charities Dataset.csv")

In [3]:
df.sort_values("Category Count", ascending=False).head(10)

,abn,Charity Name,Registration Stafus,Charity Size,staff_casual,Total Revenue,Total Expenses,Net Surplus/Deficit,Right_charity_name,classification,Program Count,Category Count,Vulnerability Score,Average Vulnerability,Total Staff,Financial Stress,Capacity Score
9091,47836436168,United Nations Association Of Australia Incorp...,Registered,Small,0,77814,161339,-83525,United Nations Association Of Australia Incorp...,Multilateral cooperation,9,29,8,0.888889,1,-1.073393,0.500000
22381,30127305127,Flowerdale Community House Inc,Registered,Small,1,151273,139136,12137,Flowerdale Community House Inc,Community development,10,28,8,0.800000,3,0.080232,0.333333
2500,86446474706,Laidley District Historical Society Inc,Registered,Small,0,16064,21196,-5132,Laidley District Historical Society Inc,Arts and culture,2,28,8,4.000000,0,-0.319472,1.000000
8793,22166587364,Centre for Organic Research and Education Limited,Registered,Small,1,160155,141383,18772,Centre for Organic Research and Education Limited,Climate change,1,28,8,8.000000,3,0.117211,0.333333
22202,54120979809,GoodCompany Foundation,Registered,Large,0,3660965,3660965,0,GoodCompany Foundation,Philanthropy promotion,1,28,8,8.000000,1,0.000000,0.500000
32994,16160276319,New Life Assembly Of God Inc,Registered,Small,0,19939,17683,2256,New Life Assembly Of God Inc,Christianity,1,28,8,8.000000,0,0.113145,1.000000
631,23167088753,Islamic Help Australia,Registered,Medium,1,456623,265126,191497,Islamic Help Australia,International development,1,28,8,8.000000,6,0.419377,0.166667
14321,92520500557,Ambassadors Of Jesus Inc,Registered,Small,5,98809,94152,4657,Ambassadors Of Jesus Inc,Welfare,2,28,7,3.500000,7,0.047131,0.333333
27295,63727400498,Planetary Healing Artists Association Of Austr...,Registered,Small,0,10841,11127,-286,Planetary Healing Artists Association Of Austr...,Social inclusion,2,28,8,4.000000,0,-0.026381,1.000000
26920,28982466474,Bayside Church International Inc.,Registered,Medium,1,297883,304055,-6172,Bayside Church International Inc.,Christianity,1,28,8,8.000000,2,-0.020720,0.500000


In [4]:
#oh there's a typo lol

df.rename(columns={'Registration Stafus': 'Registration Status'}, inplace=True)

In [5]:
#captions
#inspect the dataset

print(df.shape)
print(df.columns.tolist())
df.head()

(33060, 17)
['abn', 'Charity Name', 'Registration Status', 'Charity Size', 'staff_casual', 'Total Revenue', 'Total Expenses', 'Net Surplus/Deficit', 'Right_charity_name', 'classification', 'Program Count', 'Category Count', 'Vulnerability Score', 'Average Vulnerability', 'Total Staff', 'Financial Stress', 'Capacity Score']


,abn,Charity Name,Registration Status,Charity Size,staff_casual,Total Revenue,Total Expenses,Net Surplus/Deficit,Right_charity_name,classification,Program Count,Category Count,Vulnerability Score,Average Vulnerability,Total Staff,Financial Stress,Capacity Score
0,59633681240,CHRISTIAN MISSION FELLOWSHIP INTERNATIONAL (CM...,Registered,Small,0,56381,29933,26455,CHRISTIAN MISSION FELLOWSHIP INTERNATIONAL (CM...,Religion,1,6,1,1.0,0,0.469218,1.000000
1,80605017016,Volunteers to Assist Children with Disabilitie...,Registered,Small,0,130874,33743,97131,Volunteers to Assist Children with Disabilitie...,Special needs education,1,8,2,2.0,8,0.742172,0.111111
2,96215276030,Disabled Surfers Association Of Aust,Registered,Small,0,220277,213042,7235,Disabled Surfers Association Of Aust,Community recreation,1,2,1,1.0,0,0.032845,1.000000
3,53441697264,Riding For The Disabled Association NSW Cowra ...,Registered,Small,0,30559,29519,1040,Riding For The Disabled Association NSW Cowra ...,Animal Therapy,1,1,1,1.0,0,0.034033,1.000000
4,91111111384,Christian Family Centre Inc_ACNC Group,Registered,Large,0,2611667,2138608,473059,Christian Family Centre Inc_ACNC Group,Christianity,1,20,8,8.0,29,0.181133,0.033333


In [6]:
#captions
#clean column names in case Alteryx exported them with spaces or mixed case

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(r"[^\w]+", "_", regex=True)
)

print(df.columns.tolist())

['abn', 'charity_name', 'registration_status', 'charity_size', 'staff_casual', 'total_revenue', 'total_expenses', 'net_surplus_deficit', 'right_charity_name', 'classification', 'program_count', 'category_count', 'vulnerability_score', 'average_vulnerability', 'total_staff', 'financial_stress', 'capacity_score']


In [7]:
#quick validation of required columns before scoring

required_cols = [
    "total_staff",
    "net_surplus_deficit",
    "total_revenue",
    "category_count",
    "vulnerability_score",
    "charity_size"
]

for col in required_cols:
    if col not in df.columns:
        print(f"Missing column: {col}")

In [8]:
#convert score-related columns to numeric

numeric_cols = [
    "total_staff",
    "total_revenue",
    "net_surplus_deficit",
    "category_count",
    "vulnerability_score"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

In [9]:
#inspect minimum values before filtering

print(df[["total_staff", "total_revenue"]].describe())
df.sort_values("total_revenue")[["charity_name", "total_staff", "total_revenue"]].head(10)
df.sort_values("total_staff")[["charity_name", "total_staff", "total_revenue"]].head(10)

        total_staff  total_revenue
count  33060.000000   3.306000e+04
mean      17.868754   1.963999e+06
std       62.770475   6.075841e+06
min        0.000000   1.000000e+00
25%        0.000000   3.155025e+04
50%        1.000000   1.515835e+05
75%        7.000000   8.231062e+05
max     2938.000000   6.153947e+07


,charity_name,total_staff,total_revenue
0,CHRISTIAN MISSION FELLOWSHIP INTERNATIONAL (CM...,0,56381
18105,Family Development Foundation Australia Ltd,0,5349
18103,St John Lutheran Church Crystal Brook,0,19484
18101,Shell Cove Public School P&C Association,0,16220
18099,The Trustee For Aegium Foundation,0,335665
18096,Ashraful Madaaris Incorported,0,2500
18095,Queenstown Slavic Independent Pentecostal Chur...,0,166112
18093,Buoyancy Foundation Inc,0,64660
18107,North East Animal Sanctuary Tasmania,0,119889
18091,The Trustee For Peninsula School Foundation Ed...,0,1816165


The initial model over-prioritized extremely small charities due to high relative need, so I introduced a viability constraint to ensure recommendations focused on organizations with sufficient operational scale to effectively utilize analytical support.”

In [11]:
#remove charities too small to meaningfully absorb analyst support

df = df[
    (df["total_revenue"] >= 100000) &
    (df["total_staff"] >= 3)
].copy()

In [12]:
#confirm tiny charities are gone

print(df.shape)
df.sort_values("total_revenue")[["charity_name", "total_staff", "total_revenue"]].head(10)
df.sort_values("total_staff")[["charity_name", "total_staff", "total_revenue"]].head(10)

(11916, 17)


,charity_name,total_staff,total_revenue
23980,Youth Network Of Tasmania Inc,3,486929
27314,Australian Afghan Hassanian Youth Association ...,3,328687
21567,Shoalhaven Central Uniting Church,3,181510
13888,St. Nicholas Antiochian Orthodox Church,3,664300
30833,Kashrus Australasia Incorporated,3,227073
10880,Newman College Uniform Shop,3,148066
16348,Valentine Public School P and C,3,112147
2650,Strathfield North Parents And Citizens Associa...,3,124203
19229,Russian Orthodox Church Of The Intercession Of...,3,577378
6879,Connect Church Australia Incorporated,3,151493


In [13]:
#create size_score based on charity size

df["size_score"] = np.where(
    df["charity_size"].astype(str).str.strip().str.lower() == "medium", 1,
    np.where(
        df["charity_size"].astype(str).str.strip().str.lower() == "small", 0.7,
        0.3
    )
)

In [14]:
#create capacity score
#lower paid staffing capacity means higher need

df["capacity_score"] = 1 / (df["total_staff"] + 1)

In [15]:
#create financial score
#more negative financial position = higher priority
#this flips the sign so deficits become positive priority values

df["financial_score"] = -1 * (
    df["net_surplus_deficit"] / (df["total_revenue"] + 100000)
)

In [16]:
#create program complexity score
#log transformation reduces the effect of extreme values

df["program_complexity_score"] = np.log(df["category_count"] + 1)

In [17]:
#cap extreme financial values so outliers do not dominate

df["financial_score"] = df["financial_score"].clip(-1, 1)

In [18]:
#create program complexity score

df["program_complexity_score"] = np.log(df["category_count"] + 1)

In [19]:
#create impact score from vulnerability exposure

df["impact_score"] = df["vulnerability_score"]

In [20]:
#standardize score components so no one metric dominates because of scale differences

score_cols = [
    "capacity_score",
    "financial_score",
    "program_complexity_score",
    "impact_score",
    "size_score"
]

for col in score_cols:
    df[col + "_z"] = (df[col] - df[col].mean()) / df[col].std()

In [21]:
#create weighted priority score

df["priority_score"] = (
    0.25 * df["capacity_score_z"] +
    0.20 * df["financial_score_z"] +
    0.20 * df["program_complexity_score_z"] +
    0.25 * df["impact_score_z"] +
    0.10 * df["size_score_z"]
)

In [22]:
#create rank so highest priority charity is rank 1

df["rank"] = df["priority_score"].rank(ascending=False, method="dense")
df = df.sort_values("priority_score", ascending=False)

In [23]:
#review top recommendations
#round for cleaner Tableau display

df["priority_score"] = df["priority_score"].round(3)
df["total_revenue"] = df["total_revenue"].round(0)

#create priority tiers for storytelling

df["priority_tier"] = "Low"

df.loc[df["rank"] <= 10, "priority_tier"] = "Top Priority"
df.loc[(df["rank"] > 10) & (df["rank"] <= 50), "priority_tier"] = "High"
df.loc[(df["rank"] > 50) & (df["rank"] <= 200), "priority_tier"] = "Medium"

df[[
    "rank",
    "charity_name",
    "abn",
    "charity_size",
    "total_staff",
    "total_revenue",
    "category_count",
    "vulnerability_score",
    "priority_score"
]].head(20)

,rank,charity_name,abn,charity_size,total_staff,total_revenue,category_count,vulnerability_score,priority_score
325,1.0,Australian Guild Of Music Education Incorporated,84413249965,Medium,6,250353,20,8,2.484
10796,2.0,The Trustee For Sisters Of Charity Foundation ...,99342873237,Medium,5,710448,18,6,2.306
31036,3.0,OneSight EssilorLuxottica Foundation,34008540547,Medium,4,468257,18,6,2.261
30123,4.0,WA RETURN RECYCLE RENEW LTD,43629983615,Small,21,177993,26,8,2.144
26069,5.0,Greek Orthodox Community And Parish Of Saint G...,69045286748,Medium,14,756617,21,7,2.140
2684,6.0,Queensland Radio For The Print Handicapped Lim...,22010232934,Medium,5,283951,22,6,2.122
3603,7.0,Technology for Ageing and Disability (ACT) In...,57265712346,Small,4,222515,17,5,2.096
31505,8.0,ECSTRA FOUNDATION LIMITED,16625525162,Large,5,1092706,18,6,2.095
12353,9.0,Presentation Sisters Property Association,27009483372,Medium,5,669429,14,4,2.027
18585,10.0,The Goldfields Indigenous Housing Organ Inc.,11345673368,Medium,7,770149,15,5,2.021


In [24]:
#sanity check smallest remaining charities

df.sort_values("total_revenue")[[
    "charity_name", "total_staff", "total_revenue", "priority_score"
]].head(20)

,charity_name,total_staff,total_revenue,priority_score
24261,Australian Institute of Food Safety Foundation...,3,100000,-0.036
13473,Carmel Beit Midrash Incorporated,3,100000,0.484
30904,Yass Public School P & C Association Inc,3,100001,0.244
21709,Denmark High School Parents And Citizens Assoc...,3,100108,0.192
2850,Green Cross Project Inc.,8,100802,0.475
25598,Chabad House Of Caulfield Pty Ltd,3,101301,1.221
22749,Theodore Community Link Incorporated,7,101491,1.015
29812,Alice Springs Beanie Festival,5,101540,0.563
22288,Baptist Church New Lambton,3,101830,0.479
19844,Wangaratta Presbyterian Church Federal Board,10,101911,-0.388


In [25]:
#export final scored dataset

df.to_csv("final_scored_dataset.csv", index=False)